# Downstream Beam Prediction 

## 1. Path setup

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "src").is_dir() and (parent / "data").is_dir():
            return parent
    return start.parent


REPO_ROOT = _find_repo_root(Path.cwd())
DATA_DIR    = REPO_ROOT / "data" / "Scenario5"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
CSV_DIR     = RESULTS_DIR / "csv"
CSV_DIR.mkdir(parents=True, exist_ok=True)

for _p in (REPO_ROOT, REPO_ROOT / "src"):
    _sp = str(_p.resolve())
    if _p.exists() and _sp not in sys.path:
        sys.path.insert(0, _sp)

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR :", DATA_DIR)

REPO_ROOT: /mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets
DATA_DIR : /mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/data/Scenario5


## 2. Load clean data and the aligned `seq_index` groups

The grouping array MUST be in the same row order as `clean_data`. We read `seq_index`
from the same CSV and assert lengths match. **If the assert fails**, `load_scenario5_clean`
drops or reorders rows — replicate that filtering on `seq_groups` before continuing.

In [2]:
import numpy as np
import pandas as pd

from common.scenario5.scenario5_data import load_scenario5_clean

clean_data = load_scenario5_clean(DATA_DIR)

_df_raw = pd.read_csv(DATA_DIR / "scenario5.csv")
_df_raw = _df_raw.loc[:, ~_df_raw.columns.str.contains("^Unnamed")]
seq_groups = _df_raw["seq_index"].to_numpy()

assert len(seq_groups) == len(clean_data), (
    f"MISALIGNED: seq_groups has {len(seq_groups)} rows but clean_data has "
    f"{len(clean_data)}. load_scenario5_clean likely filters/reorders rows; "
    f"replicate that operation on seq_groups before continuing."
)

n_sequences = len(np.unique(seq_groups))
_counts = pd.Series(seq_groups).value_counts()
print(f"Rows: {len(clean_data)}  |  distinct sequences: {n_sequences}")
print(f"Rows per sequence (min/median/max): "
      f"{_counts.min()} / {int(_counts.median())} / {_counts.max()}")

Parsing location files...
Parsing mmWave power files...
Argmax matches unit1_beam_index: 100.00%
Clean data shape: (2300, 70)
Missing values after parsing: 0
Rows: 2300  |  distinct sequences: 29
Rows per sequence (min/median/max): 49 / 76 / 127


## 3. Drop unnecessary columns

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

BEAM_COLS = [f"beam_{i:02d}" for i in range(64)]
GPS_COLS = [
    "unit2_lat", "unit2_lon", "unit2_direction",
    "unit2_num_sat", "unit2_PDOP", "unit2_HDOP",
]
DOWNSTREAM_GPS_COLS = ["unit2_lat", "unit2_lon"]
RETAINED_COLS = BEAM_COLS + GPS_COLS

# Ensure we have a proper DataFrame
clean_df = (clean_data[RETAINED_COLS].copy()
            if isinstance(clean_data, pd.DataFrame)
            else pd.DataFrame(clean_data, columns=RETAINED_COLS))

scaler = MinMaxScaler()
clean_df[DOWNSTREAM_GPS_COLS] = scaler.fit_transform(clean_df[DOWNSTREAM_GPS_COLS].astype(float))

data_scaled = clean_df.to_numpy(dtype=float)

# Indices mapping remains perfectly valid
beam_idx = np.array([RETAINED_COLS.index(c) for c in BEAM_COLS])
gps_idx  = np.array([RETAINED_COLS.index(c) for c in DOWNSTREAM_GPS_COLS])

assert set(gps_idx.tolist()).isdisjoint(set(beam_idx.tolist())), (
    "LEAKAGE: gps_idx and beam_idx overlap — predictor features include label-source columns."
)

print("data_scaled:", data_scaled.shape)
print("beam_idx:", beam_idx.min(), "..", beam_idx.max(), "| gps_idx:", gps_idx.tolist())
print("Leakage guard passed: GPS features disjoint from beam label-source.")

data_scaled: (2300, 70)
beam_idx: 0 .. 63 | gps_idx: [64, 65]
Leakage guard passed: GPS features disjoint from beam label-source.


## 4. Predictor + Top-K metric 

In [4]:
import torch
import torch.nn as nn


def predict_topk_mlp(gps_train, labels_train, gps_test,
                     n_classes=64, hidden=256, n_layers=3,
                     epochs=60, lr=1e-2, batch_size=32,
                     val_frac=0.25, seed=0, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    torch.manual_seed(seed)
    np.random.seed(seed)

    y = labels_train.astype(int) - 1  

    n = len(gps_train)
    perm = np.random.default_rng(seed).permutation(n)
    n_val = max(1, int(round(val_frac * n)))
    val_i, tr_i = perm[:n_val], perm[n_val:]

    Xtr = torch.tensor(gps_train[tr_i], dtype=torch.float32, device=device)
    ytr = torch.tensor(y[tr_i],         dtype=torch.long,    device=device)
    Xva = torch.tensor(gps_train[val_i], dtype=torch.float32, device=device)
    yva = torch.tensor(y[val_i],         dtype=torch.long,    device=device)
    Xte = torch.tensor(gps_test,         dtype=torch.float32, device=device)

    layers, in_dim = [], gps_train.shape[1]
    for _ in range(n_layers):
        layers += [nn.Linear(in_dim, hidden), nn.ReLU()]
        in_dim = hidden
    layers += [nn.Linear(in_dim, n_classes)]
    model = nn.Sequential(*layers).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[20, 40], gamma=0.2)
    loss_fn = nn.CrossEntropyLoss()

    best_state, best_val_acc = None, -1.0
    idx = np.arange(len(tr_i))
    for epoch in range(epochs):
        model.train()
        np.random.shuffle(idx)
        for s in range(0, len(idx), batch_size):
            b = idx[s:s + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(Xtr[b]), ytr[b])
            loss.backward()
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_acc = (model(Xva).argmax(1) == yva).float().mean().item()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(Xte).cpu().numpy()

    ranked_idx = np.argsort(-logits, axis=1)
    return ranked_idx + 1


def topk_accuracy(ranked_labels, true_labels, top_k=(1, 2, 3, 4, 5)):
    out = {}
    for k in top_k:
        in_topk = np.any(ranked_labels[:, :k] == true_labels[:, None], axis=1)
        out[f"top{k}_acc"] = float(in_topk.mean())
    return out

## 5. Grouped downstream evaluation

Takes a `train_idx`/`test_idx` pair already produced by a grouped splitter, so the same
function serves both a single `GroupShuffleSplit` and each `GroupKFold` fold.
`imputer=None` is the **clean baseline** (train labels from un-amputed beams).

In [5]:
def _fill_residual_nans(imputed, reference_masked):
    """Belt-and-braces: if an imputer leaves NaNs, fill with column means."""
    if np.isnan(imputed).any():
        col_means = np.nanmean(reference_masked, axis=0)
        col_means = np.where(np.isnan(col_means), 0.0, col_means)
        nan_pos = np.isnan(imputed)
        imputed[nan_pos] = np.take(col_means, np.where(nan_pos)[1])
    return imputed
 
 
def downstream_eval(data_scaled, beam_idx, gps_idx, mask, imputer,
                    seed, train_idx, test_idx, train_imputed=None):
    if imputer is None:
        train_beams = data_scaled[train_idx][:, beam_idx]
    elif train_imputed is not None:
        assert len(train_imputed) == len(train_idx), (
            f"train_imputed has {len(train_imputed)} rows but train_idx has "
            f"{len(train_idx)} -- it must be train-rows-only, in train_idx order."
        )
        train_beams = train_imputed[:, beam_idx]
    else:
        data_train_masked = data_scaled[train_idx].copy()
        data_train_masked[mask[train_idx]] = np.nan
        data_train_imputed = imputer.fit_transform(data_train_masked, seed=seed)
        data_train_imputed = _fill_residual_nans(data_train_imputed, data_train_masked)
        train_beams = data_train_imputed[:, beam_idx]
 
    test_beams_clean = data_scaled[test_idx][:, beam_idx]
    labels_train = np.argmax(train_beams,      axis=1) + 1
    labels_test  = np.argmax(test_beams_clean, axis=1) + 1
 
    gps_train = data_scaled[train_idx][:, gps_idx]
    gps_test  = data_scaled[test_idx][:,  gps_idx]
 
    ranked = predict_topk_mlp(gps_train, labels_train, gps_test, seed=seed)
    return topk_accuracy(ranked, labels_test, top_k=(1, 2, 3, 4, 5))

## 6. Splitter that adapts to sequence count

Few sequences → `GroupKFold` (each sequence is test exactly once; average folds).
Many → a single `GroupShuffleSplit`. Threshold editable.

In [6]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

FEW_SEQ_THRESHOLD = 40   


def grouped_splits(groups, seed, n_folds=3, test_size=0.2):
    n_rows = len(groups)
    n_seq = len(np.unique(groups))
    all_idx = np.arange(n_rows)
    
    if n_folds == 1 or n_seq < 2:
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        tr, te = next(gss.split(all_idx, groups=groups))
        assert set(groups[tr]).isdisjoint(set(groups[te]))
        yield tr, te
        return

    if n_seq <= FEW_SEQ_THRESHOLD:
        k = min(n_folds, n_seq)
        gkf = GroupKFold(n_splits=k)
        for tr, te in gkf.split(all_idx, groups=groups):
            assert set(groups[tr]).isdisjoint(set(groups[te]))
            yield tr, te
    else:
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        tr, te = next(gss.split(all_idx, groups=groups))
        assert set(groups[tr]).isdisjoint(set(groups[te]))
        yield tr, te


print(f"Splitter: {'GroupKFold' if n_sequences <= FEW_SEQ_THRESHOLD else 'GroupShuffleSplit'} "
      f"({n_sequences} sequences).")


Splitter: GroupKFold (29 sequences).


## 7. Build masks for each (mechanism, proportion)

The mask is dataset-level; the grouped split only changes which *rows* are train.
The MCAR builder is inline; for MAR/MNAR, wire in the project's helper (restricted to
`beam_idx`) where indicated.

In [7]:
from common.amputation import build_mask

MECHANISMS  = ["MCAR", "MAR", "MNAR"]
PROPORTIONS = [0.10, 0.30, 0.50]
SEED = 0

mar_driver_idx = [RETAINED_COLS.index("unit2_HDOP")]   

_rng = np.random.default_rng(SEED)
_m = build_mask("MCAR", data_scaled, 0.10, beam_idx, _rng,
                driver_col_idx=mar_driver_idx, mar_score_mode="single")
print("Realised beam missingness:", round(float(_m[:, beam_idx].mean()), 4), "(target 0.10)")
assert not _m[:, gps_idx].any(), "GPS columns were amputed — build_mask not restricted to beams."
print("GPS columns untouched by mask: OK")


Realised beam missingness: 0.1012 (target 0.10)
GPS columns untouched by mask: OK


## 8. Run: clean baseline vs imputers under the grouped split

In [ ]:
import time
from imputers.imputers import (
    MeanImputer, KNNImputerWrapper, SoftImputeWrapper, MICEImputer,
    HyperImputeImputer,
)
from imputers.diffputer_imputer import DiffPuterImputer
from imputers.grape_imputer import GRAPEImputer
from common.tuning import load_tuned_params

# ---------------------------------------------------------------------------
# Tuned hyperparameters.
# Load the SAME config used in the reconstruction experiment so the two tables
# stay comparable (one frozen pipeline, evaluated two ways). We index keys
# directly and assert presence: a missing tuned value should fail loudly, not
# silently fall back to a library default.
# ---------------------------------------------------------------------------
TUNED_PARAMS_PATH = RESULTS_DIR  / "tuned_params.json"
tp = load_tuned_params(TUNED_PARAMS_PATH, "scenario5")
assert tp, (
    f"No tuned params for 'scenario5' in {TUNED_PARAMS_PATH}. "
    "Run scenario5_tuning.ipynb first."
)
for _k in ("knn_n_neighbors", "softimpute_shrinkage", "diffputer",
           "grape_epochs", "grape_node_edge_dim", "grape_lr"):
    assert _k in tp, f"tuned_params.json missing key '{_k}' for scenario5."
print("Loaded tuned params:",
      {k: ("{...}" if isinstance(v, dict) else v) for k, v in tp.items()})

# ---------------------------------------------------------------------------
# Run plan. Same RMSE-tuned hyperparameters for every method; HyperImpute added
# for parity with the reconstruction table (it self-tunes, no external params).
# Tuple: (label, imputer, mechanisms, proportions, n_folds, per_fold_refit)
# ---------------------------------------------------------------------------
FULL_MECH = ["MCAR", "MAR", "MNAR", "MCAR-Row"]
FULL_PROP = [0.10, 0.30, 0.50]
 
IMPUTERS = [
    ("mean",        MeanImputer(),                                                 FULL_MECH, FULL_PROP, 3),
    ("knn",         KNNImputerWrapper(k=tp["knn_n_neighbors"]),                    FULL_MECH, FULL_PROP, 3),
    ("softimpute",  SoftImputeWrapper(shrinkage_value=tp["softimpute_shrinkage"]), FULL_MECH, FULL_PROP, 3),
    ("mice",        MICEImputer(),                                                 FULL_MECH, FULL_PROP, 1),
    ("hyperimpute", HyperImputeImputer(),                                          FULL_MECH, FULL_PROP, 1),
    ("diffputer",   DiffPuterImputer(**tp["diffputer"]),                           FULL_MECH, FULL_PROP, 1), 
    ("grape",       GRAPEImputer(epochs=tp["grape_epochs"],
                                 node_dim=tp["grape_node_edge_dim"],
                                 edge_dim=tp["grape_node_edge_dim"],  
                                 lr=tp["grape_lr"]),                               FULL_MECH, FULL_PROP, 1), 
]
 
SEEDS = (0, 1, 2)
CLEAN_FOLDS = 3
 
DOWNSTREAM_PATH = CSV_DIR / "scenario5_downstream_full_results_new.csv"
CACHE_DIR = RESULTS_DIR / "scenario5" / "imputed_cache_new"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _cache_path(method, mech, prop, seed, fold):
    # fold IS part of the key now: each fold imputes its own train rows, so the
    # cached artifact is a per-fold TRAIN-ONLY imputation. The old full-array
    # files (no __f) are stale -- delete them before rerunning.
    return CACHE_DIR / f"{method}__{mech}__p{int(round(prop * 100)):02d}__s{seed}__f{fold}.npy"


def get_train_imputation(imputer, method, mech, prop, mask, seed, train_idx, fold):
    cpath = _cache_path(method, mech, prop, seed, fold)
    if cpath.exists():
        cached = np.load(cpath)
        if cached.shape[0] == len(train_idx):
            return cached
        cpath.unlink(missing_ok=True)
    data_train_masked = data_scaled[train_idx].copy()
    data_train_masked[mask[train_idx]] = np.nan
    t0 = time.time()
    imputed = imputer.fit_transform(data_train_masked, seed=seed)
    imputed = _fill_residual_nans(imputed, data_train_masked)
    np.save(cpath, imputed)
    print(f"  [cache] imputed {method} {mech} p={prop} s={seed} f{fold} "
          f"in {time.time() - t0:.1f}s -> {cpath.name}")
    return imputed
 
 
def _agg(cell_runs, base_row):
    """Mean/std over all (seed, fold) replicates for one result cell."""
    row = dict(base_row)
    acc_keys = [k for k in cell_runs[0] if k.endswith("_acc")]
    row["n_runs"] = len(cell_runs)
    for k in acc_keys:
        vals = [r[k] for r in cell_runs]
        row[k] = float(np.mean(vals))
        row[f"{k}_std"] = float(np.std(vals))
    return row
 
 
# Resume checkpoint (granularity: mechanism x proportion x method).
if DOWNSTREAM_PATH.exists():
    done_df = pd.read_csv(DOWNSTREAM_PATH)
    done_keys = set(zip(done_df["mechanism"], done_df["proportion"], done_df["method"]))
    records = done_df.to_dict("records")
    print(f"Resuming: {len(done_keys)} result cells already in {DOWNSTREAM_PATH.name}.")
else:
    done_keys, records = set(), []
 
 
def _append_downstream(row):
    pd.DataFrame([row]).to_csv(
        DOWNSTREAM_PATH, mode="a", header=not DOWNSTREAM_PATH.exists(), index=False
    )


# ---------------------------------------------------------------------------
# CLEAN baseline: mask-independent, but still repeated over seeds (varies the
# split RNG + MLP init) so it carries an error bar consistent with the rest.
# Computed once, then broadcast to every (mechanism, proportion) panel.
# ---------------------------------------------------------------------------
clean_needed = [(m, p) for m in FULL_MECH for p in FULL_PROP
                if (m, p, "clean") not in done_keys]
 
if clean_needed:
    t0 = time.time()
    clean_runs = []
    for seed in SEEDS:
        for tr, te in grouped_splits(seq_groups, seed=seed, n_folds=CLEAN_FOLDS):
            clean_runs.append(downstream_eval(
                data_scaled, beam_idx, gps_idx, mask=None, imputer=None,
                seed=seed, train_idx=tr, test_idx=te, train_imputed=None,
            ))
    clean_stats = _agg(clean_runs, {"cached": False, "secs": round(time.time() - t0, 1)})
    for mech, prop in clean_needed:
        row = {"mechanism": mech, "proportion": prop, "method": "clean", "n_folds": CLEAN_FOLDS, **clean_stats}
        records.append(row)
        done_keys.add((mech, prop, "clean"))
        _append_downstream(row)
    print(f"clean (over {len(SEEDS)} seeds, {clean_stats['n_runs']} runs, "
          f"{clean_stats['secs']}s) broadcast to {len(clean_needed)} cells: "
          f"top1={clean_stats['top1_acc']:.3f} top5={clean_stats['top5_acc']:.3f}")


# ---------------------------------------------------------------------------
# Imputers: the mask matters, so loop mechanism x proportion x seed.
# Within a seed the imputation is computed once (cached) and sliced per fold;
# flip per_fold_refit=True for a method to impute on train rows only (strictly
# inductive, no transductive peek at test rows).
# --imputation_env-------------------------------------------------------------------------
for label, imp, mechs, props, n_folds in IMPUTERS:
    for mech in mechs:
        for prop in props:
            key = (mech, prop, label)
            if key in done_keys:
                continue
 
            t0 = time.time()
            cell_runs = []
            for seed in SEEDS:
                rng = np.random.default_rng(seed)
                mask = build_mask(mech, data_scaled, prop, beam_idx, rng,
                                  driver_col_idx=mar_driver_idx,
                                  mar_score_mode="single")
 
                for fold, (tr, te) in enumerate(
                        grouped_splits(seq_groups, seed=seed, n_folds=n_folds)):
                    train_imputed = None
                    if imp is not None:
                        train_imputed = get_train_imputation(
                            imp, label, mech, prop, mask, seed, tr, fold)
                    cell_runs.append(downstream_eval(
                        data_scaled, beam_idx, gps_idx, mask, imp,
                        seed=seed, train_idx=tr, test_idx=te,
                        train_imputed=train_imputed,
                    ))
 
            row = _agg(cell_runs, {
                "mechanism": mech, "proportion": prop, "method": label,
                "cached": True, "n_folds": n_folds,
                "secs": round(time.time() - t0, 1),
            })
            records.append(row)
            done_keys.add(key)
            _append_downstream(row)
            print(f"{mech} p={prop} {label:11s} "
                  f"top1={row['top1_acc']:.3f}±{row['top1_acc_std']:.3f} "
                  f"top5={row['top5_acc']:.3f}  "
                  f"({row['n_runs']} runs, {n_folds}-fold cached, "
                  f"{row['secs']}s)")
 
results = pd.DataFrame(records)
results

Loaded tuned params: {'knn_n_neighbors': 5, 'softimpute_shrinkage': None, 'grape_epochs': 20000, 'grape_node_edge_dim': 128, 'grape_lr': 0.000904395708283654, 'diffputer': '{...}'}
Resuming: 90 result cells already in scenario5_downstream_full_results_new.csv.


## 9. Clean vs imputed — the actual comparison


In [ ]:
metric = "top1_acc"
pivot = results.pivot_table(index=["mechanism", "proportion"],
                            columns="method", values=metric)
if "clean" in pivot.columns:
    for col in list(pivot.columns):
        if col != "clean":
            pivot[f"{col}_minus_clean"] = pivot[col] - pivot["clean"]

results.to_csv(CSV_DIR / "scenario5_downstream_grouped_new.csv", index=False)
print("Saved:", CSV_DIR / "scenario5_downstream_grouped_new.csv")
pivot.round(4)


In [ ]:
"""
Scenario-5 downstream Top-K plots.
"""
import matplotlib.pyplot as plt
from common.visualization import METHOD_COLORS

LABEL_TO_DISPLAY = {
    "clean": "Clean", "mean": "Mean", "knn": "kNN", "mice": "MICE",
    "softimpute": "SoftImpute", "hyperimpute": "HyperImpute",
    "grape": "GRAPE", "diffputer": "DiffPuter",
}
IMPUTER_METHODS = ["Mean", "kNN", "MICE", "HyperImpute",
                   "SoftImpute", "GRAPE", "DiffPuter"]


def _as_plot_df(data):
    df = data if isinstance(data, pd.DataFrame) else pd.read_csv(data)
    df = df.copy()
    df["method"] = df["method"].map(lambda m: LABEL_TO_DISPLAY.get(m, m))
    if "scenario" not in df.columns and "mechanism" in df.columns:
        df["scenario"] = df["mechanism"]
    return df


def _present_imputers(sub):
    present = set(sub["method"].unique())
    return [m for m in IMPUTER_METHODS if m in present]


def _draw_topk_panel(ax, sub, methods, show_ylabel, show_legend):
    ks = [1, 2, 3, 4, 5]
    metrics = [f"top{k}_acc" for k in ks]
    bar_width = 0.8 / max(len(methods), 1)
    group_centers = np.arange(len(ks))

    for i, m in enumerate(methods):
        msub = sub[sub["method"] == m]
        means = [msub[met].mean() for met in metrics]
        # Prefer the stored fold std if present, else std across rows.
        stds = []
        for met in metrics:
            std_col = f"{met}_std"
            stds.append(msub[std_col].mean() if std_col in msub else msub[met].std())
        offsets = group_centers - 0.4 + (i + 0.5) * bar_width
        ax.bar(offsets, means, width=bar_width, yerr=stds, label=m,
               color=METHOD_COLORS.get(m), edgecolor="black", linewidth=0.4,
               error_kw=dict(elinewidth=0.7, capsize=1.5, ecolor="black"),
               alpha=0.9)

    clean = sub[sub["method"] == "Clean"]
    if not clean.empty:
        tick_half = bar_width * len(methods) / 2 + bar_width * 0.3
        for j, met in enumerate(metrics):
            ax.hlines(clean[met].mean(),
                      group_centers[j] - tick_half, group_centers[j] + tick_half,
                      color="black", linestyle="--", linewidth=1.1,
                      label="Clean (ceiling)" if j == 0 else None, zorder=10)

    ax.set_xticks(group_centers)
    ax.set_xticklabels([f"Top-{k}" for k in ks])
    ax.set_xlabel("Candidate set size")
    if show_ylabel:
        ax.set_ylabel("Accuracy")
    ax.grid(axis="y", alpha=0.25, linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if show_legend:
        ax.legend(loc="lower right", ncol=2, fontsize=8, frameon=False)


def plot_topk_comparison_grid(data, scenario="MAR", proportions=None,
                              filename=None, methods=None):
    df = _as_plot_df(data)
    sc = df[df["scenario"] == scenario]
    if sc.empty:
        print(f"[plot] no rows for scenario={scenario}; skipping.")
        return
    proportions = proportions or sorted(sc["proportion"].unique())
    methods = methods or _present_imputers(sc)

    fig, axes = plt.subplots(1, len(proportions),
                             figsize=(3.2 * len(proportions), 3.6),
                             sharey=True)
    if len(proportions) == 1 and not hasattr(axes, "__iter__"):
        axes = [axes]

    for ax_i, (ax, p) in enumerate(zip(axes, proportions)):
        sub = sc[sc["proportion"] == p]
        _draw_topk_panel(ax, sub, methods,
                         show_ylabel=(ax_i == 0), show_legend=False)
        ax.set_title(f"{int(p*100)}% missing")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(len(labels), 5),
               bbox_to_anchor=(0.5, -0.10), frameon=False)
    fig.suptitle(f"Top-K beam prediction under {scenario}", y=1.02)
    fig.tight_layout()
    if filename is not None:
        fig.savefig(filename, bbox_inches="tight")
    plt.show()


# ---------------------------------------------------------------------------
# Usage — note heavy methods only populate their trimmed proportions, so some
# panels will show fewer bars; that is expected from the run plan.
# ---------------------------------------------------------------------------
for mech in ["MCAR", "MAR", "MNAR", "MCAR-Row"]:
    plot_topk_comparison_grid(results, scenario=mech,
                              filename=FIGURES_DIR / f"sq3_topk_grid_{mech}.png")
